#### Unity AI Gateway Demo

Demonstrates the four governance pillars of Unity AI Gateway for the Casper's Kitchens
refund and complaint agents.  Both agents route their LLM calls through the gateway
endpoint named by `AI_GATEWAY_ENDPOINT_NAME` so every request is subject to the same
guardrails, logged to the same audit table, and counted against the same usage quota.

| Beat | What it shows |
|------|---------------|
| 1 — Connectivity | Gateway is reachable and the model responds |
| 2 — Guardrails | Complaints with PII (SSN / CC) are blocked before reaching the LLM |
| 3 — Audit trail | Every agent LLM call logged to a Delta inference table in UC |
| 4 — Usage tracking | Per-endpoint token consumption via `system.ai_gateway.usage` |

In [ ]:
dbutils.widgets.text("CATALOG", "")
dbutils.widgets.text("AI_GATEWAY_ENDPOINT_NAME", "")
dbutils.widgets.text("AI_GATEWAY_CATALOG", "")
dbutils.widgets.text("AI_GATEWAY_SCHEMA", "ai_gateway")

CATALOG = dbutils.widgets.get("CATALOG")
AI_GATEWAY_ENDPOINT_NAME = dbutils.widgets.get("AI_GATEWAY_ENDPOINT_NAME")
AI_GATEWAY_CATALOG = dbutils.widgets.get("AI_GATEWAY_CATALOG") or CATALOG
AI_GATEWAY_SCHEMA = dbutils.widgets.get("AI_GATEWAY_SCHEMA") or "ai_gateway"

if not AI_GATEWAY_ENDPOINT_NAME:
    raise ValueError(
        "AI_GATEWAY_ENDPOINT_NAME is required for this stage. "
        "Configure a Unity AI Gateway endpoint in the Databricks UI and pass "
        "--params 'AI_GATEWAY_ENDPOINT_NAME=<endpoint-name>' when running the 'all' target. "
        "See demos/dais2026-runbooks/SETUP.ipynb for step-by-step instructions."
    )

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
HOST = w.config.host.rstrip("/")
TOKEN = w.config.token
GATEWAY_URL = f"{HOST}/ai-gateway/mlflow/v1/chat/completions"
INFERENCE_TABLE = f"`{AI_GATEWAY_CATALOG}`.`{AI_GATEWAY_SCHEMA}`.`{AI_GATEWAY_SCHEMA}_payload`"

print(f"Catalog:           {CATALOG}")
print(f"Gateway endpoint:  {AI_GATEWAY_ENDPOINT_NAME}")
print(f"Gateway URL:       {GATEWAY_URL}")
print(f"Inference table:   {INFERENCE_TABLE}")

#### Beat 1 — Connectivity

Verify the gateway endpoint is reachable and returns a valid response.

In [ ]:
import requests

def _gateway_request(messages, max_tokens=64):
    return requests.post(
        GATEWAY_URL,
        headers={"Authorization": f"Bearer {TOKEN}"},
        json={"model": AI_GATEWAY_ENDPOINT_NAME, "messages": messages, "max_tokens": max_tokens},
        timeout=60,
    )

resp = _gateway_request([{"role": "user", "content": "Say 'gateway ok' and nothing else."}])
print(f"HTTP {resp.status_code}")
if resp.status_code == 200:
    content = resp.json()["choices"][0]["message"]["content"]
    usage = resp.json().get("usage", {})
    print(f"Response: {content}")
    print(f"Tokens:   {usage.get('total_tokens', '?')} total ({usage.get('prompt_tokens','?')} in / {usage.get('completion_tokens','?')} out)")
    print(f"\n✅ Gateway is reachable — endpoint: {AI_GATEWAY_ENDPOINT_NAME}")
else:
    print(f"Body: {resp.text[:500]}")
    raise RuntimeError(
        f"Gateway returned HTTP {resp.status_code}. "
        "Verify the endpoint is READY and AI Gateway is enabled on it."
    )

#### Beat 2 — Guardrails: PII Blocking

Customer complaints sometimes contain PII (SSNs, credit card numbers, phone numbers).
With PII detection in **BLOCK** mode, the gateway rejects the request before it reaches
the LLM — the agent never sees the sensitive data.

In [ ]:
PII_SCENARIOS = [
    {
        "name": "SSN in complaint",
        "expected": "blocked",
        "messages": [{"role": "user", "content": (
            "My order ORD-99999 was cold and late! "
            "My social security number is 123-45-6789. I want a full refund."
        )}],
    },
    {
        "name": "Credit card in request",
        "expected": "blocked",
        "messages": [{"role": "user", "content": (
            "Order ORD-99998 never arrived. "
            "Bill the refund to my card 4111-1111-1111-1111 expiry 12/26."
        )}],
    },
    {
        "name": "Clean complaint (no PII)",
        "expected": "allowed",
        "messages": [{"role": "user", "content": (
            "My pizza in order ORD-99997 arrived 45 minutes late and was cold. "
            "Please issue a partial refund."
        )}],
    },
]

passed = 0
print("Guardrail test results")
print("=" * 62)
for s in PII_SCENARIOS:
    r = _gateway_request(s["messages"], max_tokens=20)
    actual = "blocked" if r.status_code != 200 else "allowed"
    ok = actual == s["expected"]
    passed += ok
    icon = "✅" if ok else "❌"
    outcome_tag = "BLOCKED" if actual == "blocked" else "ALLOWED"
    print(f"  {icon} [{outcome_tag:7s}] HTTP {r.status_code}  {s['name']}")

print()
if passed == len(PII_SCENARIOS):
    print(f"✅ All {passed}/{len(PII_SCENARIOS)} guardrail tests passed.")
else:
    print(f"⚠️  {passed}/{len(PII_SCENARIOS)} passed. Check guardrail config in the AI Gateway UI.")

#### Beat 3 — Audit Trail

Every request that flows through the gateway — allowed or blocked — is written to a
Delta inference table in Unity Catalog.  This gives a complete, queryable audit log
of every LLM call the agents made, including the full request and response payloads.

In [ ]:
try:
    display(spark.sql(f"""
        SELECT
            event_time,
            request_id,
            status_code,
            requester,
            latency_ms,
            request,
            response
        FROM {INFERENCE_TABLE}
        ORDER BY event_time DESC
        LIMIT 20
    """))
except Exception as e:
    print(f"Could not query inference table: {e}")
    print(
        f"Ensure inference tables are enabled in the AI Gateway UI "
        f"and pointed at {AI_GATEWAY_CATALOG}.{AI_GATEWAY_SCHEMA}."
    )

In [ ]:
# Zoom in on blocked requests to show the guardrail audit trail
try:
    display(spark.sql(f"""
        SELECT
            event_time,
            request_id,
            status_code,
            requester,
            latency_ms,
            logging_error_codes
        FROM {INFERENCE_TABLE}
        WHERE status_code != 200
        ORDER BY event_time DESC
        LIMIT 20
    """))
except Exception as e:
    print(f"Could not query blocked requests: {e}")

#### Beat 4 — Usage Tracking

`system.ai_gateway.usage` captures per-request token counts for every
gateway-enabled endpoint.  Slice by endpoint to see the token spend for the
current session — and in a multi-endpoint setup, attribute costs to individual agents.

In [ ]:
try:
    display(spark.sql(f"""
        SELECT
            endpoint_name,
            DATE_TRUNC('hour', usage_time)    AS hour,
            SUM(input_tokens)                 AS input_tokens,
            SUM(output_tokens)                AS output_tokens,
            SUM(input_tokens + output_tokens) AS total_tokens
        FROM system.ai_gateway.usage
        WHERE endpoint_name = '{AI_GATEWAY_ENDPOINT_NAME}'
          AND usage_time >= CURRENT_TIMESTAMP - INTERVAL 1 HOUR
        GROUP BY endpoint_name, DATE_TRUNC('hour', usage_time)
        ORDER BY hour DESC
    """))
except Exception as e:
    print(f"Could not query system.ai_gateway.usage: {e}")
    print("Ensure usage tracking is enabled in the AI Gateway UI → Usage Tracking tab.")